# Beryllene SOC topology results

Results of the current SOC topology campaign, read directly from the output files in `9.0_topology/`. All invariants are **conditional screening results** (`topology_certified: false`): each Z₂ value belongs to a fixed lowest-N spinor-band subspace and assumes electronic time reversal and isolation of that subspace. Except for 2H-α, the structures are metallic at neutral filling, so no value is a Fermi-level quantum-spin-Hall invariant.

[Calculation workflow and troubleshooting](9.0_topology/workflow.md) · [Report 2026-09-26](9.0_topology/topo_report_20260926.md)

In [1]:
from pathlib import Path
import json
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
repo = next((p for p in (cwd, *cwd.parents)
             if (p / "9.0_topology/manifest.json").is_file()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from the project directory.")
topology = repo / "9.0_topology"
structures = json.loads((topology / "manifest.json").read_text())["structures"]
labels = {"alpha": "α", "beta": "β", "st": "ST", "alpha_2h": "2H-α",
          "beta_1h": "1H-β", "beta_2h": "2H-β"}


def load(path):
    return json.loads(path.read_text()) if path.is_file() else None


def latest(folder, pattern):
    found = sorted(folder.glob(pattern))
    return found[-1] if found else None


def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def sign(value):
    return "+" if value > 0 else "−"


def fermi_level(folder):
    lines = (folder / "scf" / "DOSCAR").read_text().splitlines()
    return float(lines[5].split()[3])


def extrema(folder):
    # Final sampled extrema over scf, trim, bands and all refinement stages.
    zoom = load(folder / "gap_refine_3_summary.json")
    if zoom:
        return zoom["sampled_extrema_after"], True
    return load(folder / "gap_refinement_summary.json")["final_sampled_extrema"], False

## Summary

In [2]:
rows = []
for entry in structures:
    folder = topology / entry["directory"]
    parity_dir = latest(folder / "trim", "parity-analysis-*")
    parity = load(parity_dir / "parity_summary.json") if parity_dir else None
    wcc = load(latest(folder, "wcc_direct_v3/status_*.json")) if (folder / "wcc_direct_v3").is_dir() else None
    if parity:
        invariant = "ν = %d (parity)" % parity["conditional_fu_kane_nu"]
    elif wcc:
        invariant = ", ".join("N=%s: Z₂ = %d (WCC)" % (n, m["z2"]) for n, m in wcc["manifolds"].items())
    else:
        invariant = "not validated"
    fermi = fermi_level(folder)
    gaps, zoomed = extrema(folder)
    direct = ", ".join("%.4g eV" % g["direct_gap_ev"] if g["direct_gap_ev"] >= 0.1
                       else "%.4g meV" % (1000 * g["direct_gap_ev"]) for g in gaps)
    insulating = all(g["valence_max_ev"] < fermi < g["conduction_min_ev"] for g in gaps)
    spin = load(folder / "spin_screen" / "spin_screen_summary.json")
    tr = load(folder / "tr_evidence.json")
    rows.append([labels[entry["id"]], entry["directory"], entry["expected_nelect"],
                 invariant, direct + (" (zoom pending)" if not zoomed and entry["id"] in ("alpha", "beta", "st") else ""),
                 "insulator" if insulating else "metal",
                 "%.1e" % tr["magnetisation_density"]["max_abs_m_muB_per_A3"] if tr else "pending",
                 ("collapsed" if spin["all_seeds_collapsed"] else "MOMENT KEPT") if spin else "pending"])
table(["Structure", "Directory", "Spinor bands N (NELECT)", "Conditional invariant",
       "Min. sampled direct gap E(N+1)−E(N)", "Neutral E_F", "max \\|m(r)\\| (μB/Å³)", "Spin seeds"], rows)

| Structure | Directory | Spinor bands N (NELECT) | Conditional invariant | Min. sampled direct gap E(N+1)−E(N) | Neutral E_F | max \|m(r)\| (μB/Å³) | Spin seeds |
|---|---|---|---|---|---|---|---|
| α | a-Beryllene | 2 | ν = 1 (parity) | 1.127 meV | metal | 5.9e-08 | pending |
| β | b-Beryllene | 4 | ν = 0 (parity) | 0.854 meV | metal | 8.9e-08 | pending |
| ST | c-Beryllene_trilayer | 6 | ν = 1 (parity) | 0.735 meV | metal | 1.0e-07 | pending |
| 2H-α | a-Beryllene_hh | 4 | ν = 0 (parity) | 4.925 eV | insulator | 5.9e-08 | pending |
| 1H-β | b-Beryllene_b | 5 | N=4: Z₂ = 0 (WCC), N=6: Z₂ = 0 (WCC) | 0.9295 eV, 0.415 eV | metal | 6.1e-08 | pending |
| 2H-β | b-Beryllene_bb | 6 | ν = 0 (parity) | 0.1771 eV | metal | 2.6e-07 | pending |

## Fu–Kane parity at the four 2D TRIM

IrRep 2.1.3 characters of `{−1|τ}` on the SOC spinor WAVECAR at Γ, X=(½,0), Y=(0,½), M=(½,½) of each primitive cell. Each Kramers pair is listed from band 1 upward; the entry after `|` is the next pair above band N, and the last column gives E(N+1)−E(N) at that TRIM. The independent float64 check reconstructs the inversion matrix from the WAVECAR and must agree with IrRep (tolerances: 1e-6 for Löwdin unitarity, off-block elements and opposite-parity overlap; 1e-5 for the per-band inversion residual).

In [3]:
rows, checks = [], []
for entry in structures:
    folder = topology / entry["directory"]
    parity_dir = latest(folder / "trim", "parity-analysis-*")
    if not entry["inversion_expected"] or parity_dir is None:
        continue
    report = load(parity_dir / "parity_summary.json")
    if report is None:
        rows.append([labels[entry["id"]], "not validated", "", "", "", "", ""])
        continue
    cells = []
    for name in ("Gamma", "X", "Y", "M"):
        point, check = report["trims"][name], report["wavefunction_validation"][name]
        pairs = " ".join(sign(p) for block in point["blocks"]
                         for p in block["kramers_pair_parities_unordered_within_block"])
        above = check["next_block_above_N_parity"]
        cells.append("%s \\| %s (%.3g eV)" % (pairs, sign(above) if above else "?",
                                              check["next_block_above_N_gap_ev"]))
    deltas = " ".join(sign(report["trims"][n]["delta"]) for n in ("Gamma", "X", "Y", "M"))
    rows.append([labels[entry["id"]], report["occupied_spinor_bands"], *cells, deltas,
                 report["conditional_fu_kane_nu"]])
    values = report["wavefunction_validation"].values()
    checks.append([labels[entry["id"]], parity_dir.name,
                   "%.1e" % max(v["lowdin_inversion_singular_value_max_error"] for v in values),
                   "%.1e" % max(v["plain_metric_opposite_parity_max_overlap"] for v in values),
                   "%.1e" % max(v["inversion_residual_max"] for v in values),
                   "%.1e" % max(v["plain_metric_same_parity_max_overlap"] for v in values),
                   ", ".join("%.3g" % v for v in report["irrep_plain_metric_orthogonality_messages"]) or "none"])
table(["Structure", "N", "Γ", "X", "Y", "M", "δ (Γ X Y M)", "ν"], rows)
table(["Structure", "Analysis", "Löwdin unitarity error", "Opposite-parity overlap",
       "Inversion residual", "Same-parity overlap (PAW metric)", "IrRep plain-metric message"], checks)

| Structure | N | Γ | X | Y | M | δ (Γ X Y M) | ν |
|---|---|---|---|---|---|---|---|
| α | 2 | + \| − (5.1 eV) | − \| + (6.37 eV) | − \| + (6.37 eV) | − \| + (6.37 eV) | + − − − | 1 |
| β | 4 | + − \| + (2.81 eV) | + − \| − (5.24 eV) | + − \| − (5.95 eV) | + − \| − (5.52 eV) | − − − − | 0 |
| ST | 6 | + − + \| − (2.22 eV) | + − + \| − (5.1 eV) | + − + \| − (5.1 eV) | − − + \| + (0.00128 eV) | − − − + | 1 |
| 2H-α | 4 | + − \| + (9.79 eV) | − + \| − (5.64 eV) | − + \| − (5.64 eV) | − + \| − (5.64 eV) | − − − − | 0 |
| 2H-β | 6 | + − + \| − (8.37 eV) | − + + \| − (1.3 eV) | + − − \| + (1.3 eV) | − + − \| + (0.177 eV) | − − + + | 0 |

| Structure | Analysis | Löwdin unitarity error | Opposite-parity overlap | Inversion residual | Same-parity overlap (PAW metric) | IrRep plain-metric message |
|---|---|---|---|---|---|---|
| α | parity-analysis-20260926T041719Z-cc2e28f1 | 2.6e-12 | 1.0e-08 | 2.2e-06 | 3.5e-08 | none |
| β | parity-analysis-20260926T041738Z-b9bc8c36 | 1.5e-11 | 4.0e-08 | 5.0e-06 | 2.6e-02 | 1.1e-05, 1.2e-05 |
| ST | parity-analysis-20260926T041756Z-864aa1bf | 1.0e-11 | 2.2e-08 | 3.7e-06 | 8.5e-03 | 0.00653, 1.84e-05, 1.82e-05, 1.26e-05 |
| 2H-α | parity-analysis-20260926T041825Z-4b717816 | 1.6e-11 | 2.2e-08 | 5.7e-06 | 1.7e-02 | 3.62e-05, 3.73e-05, 3.77e-05, 4.43e-05 |
| 2H-β | parity-analysis-20260926T041844Z-7a125604 | 2.5e-11 | 2.3e-08 | 6.4e-06 | 2.0e-02 | 0.0155, 0.00798, 0.00791, 0.00286 |

## Isolation of the lowest-N subspace and the neutral Fermi level

Minimum sampled direct gap E(N+1)−E(N) over the 105×105 SCF grid, the TRIM, the band path and all local refinements; valence maximum of band N and conduction minimum of band N+1 relative to the SCF Fermi level (DOSCAR). A negative indirect gap means bands N and N+1 both cross E_F.

In [4]:
rows = []
for entry in structures:
    folder = topology / entry["directory"]
    fermi = fermi_level(folder)
    gaps, zoomed = extrema(folder)
    for g in gaps:
        rows.append([labels[entry["id"]], g["N"], "%.6g" % (1000 * g["direct_gap_ev"]),
                     "(%.5f, %.5f)" % tuple(g["direct_gap_k"][:2]), g["direct_gap_stage"],
                     "%+.3f" % (g["valence_max_ev"] - fermi), "%+.3f" % (g["conduction_min_ev"] - fermi),
                     "%+.3f" % g["indirect_gap_ev"]])
table(["Structure", "N", "Direct gap (meV)", "k (fractional)", "Stage", "max E_N − E_F (eV)",
       "min E_N+1 − E_F (eV)", "Indirect gap (eV)"], rows)

| Structure | N | Direct gap (meV) | k (fractional) | Stage | max E_N − E_F (eV) | min E_N+1 − E_F (eV) | Indirect gap (eV) |
|---|---|---|---|---|---|---|---|
| α | 2 | 1.127 | (0.33333, 0.33333) | gap_refine_3d | +2.722 | -1.307 | -4.029 |
| β | 4 | 0.854 | (0.32842, 0.31373) | gap_refine_3e | +1.617 | -1.901 | -3.518 |
| ST | 6 | 0.735 | (0.31935, 0.31935) | gap_refine_3d | +2.528 | -1.222 | -3.750 |
| 2H-α | 4 | 4924.52 | (0.43333, 0.00000) | gap_refine_1 | -0.065 | +4.776 | +4.842 |
| 1H-β | 4 | 929.475 | (0.46607, 0.00000) | gap_refine_2 | +0.083 | -3.004 | -3.087 |
| 1H-β | 6 | 415.04 | (0.21786, 0.37738) | gap_refine_2 | +3.334 | +2.311 | -1.023 |
| 2H-β | 6 | 177.064 | (-0.49524, 0.49524) | scf | +2.120 | -4.033 | -6.152 |

### Adaptive zoom into unresolved direct-gap basins (`gap_refine_3*`)

Each level is a 9×9 patch per basin, re-centred on the previous minimum and shrunk 4× when that minimum was interior. The slope bound is min gap − (largest neighbour slope) × (sampling radius): a heuristic local lower bound, positive values support but do not prove a gap.

In [5]:
rows = []
for entry in structures:
    zoom = load(topology / entry["directory"] / "gap_refine_3_summary.json")
    if zoom is None:
        if entry["id"] in ("alpha", "beta", "st"):
            rows.append([labels[entry["id"]], "pending", "", "", "", "", ""])
        continue
    for index, basin in enumerate(zoom["basins"]):
        for level in basin["levels"]:
            rows.append([labels[entry["id"]], "basin %d start (%.5f, %.5f)" % (index + 1, *basin["start"]),
                         level["stage"], "%.2e" % level["spacing"],
                         "%.4f" % (1000 * level["min_direct_gap_ev"]),
                         "(%.7f, %.7f)%s" % (*level["min_k"], "" if level["min_interior"] else " edge"),
                         "%.4f" % (1000 * level["slope_bound_ev"])])
table(["Structure", "Basin", "Stage", "Spacing (frac.)", "Min gap (meV)", "At k", "Slope bound (meV)"], rows)

| Structure | Basin | Stage | Spacing (frac.) | Min gap (meV) | At k | Slope bound (meV) |
|---|---|---|---|---|---|---|
| α | basin 1 start (0.33333, 0.33333) | gap_refine_3a | 1.49e-04 | 1.1562 | (0.3333333, 0.3333333) | -5.1734 |
| α | basin 1 start (0.33333, 0.33333) | gap_refine_3b | 3.72e-05 | 1.1562 | (0.3333333, 0.3333333) | -0.4047 |
| α | basin 1 start (0.33333, 0.33333) | gap_refine_3c | 9.30e-06 | 1.1557 | (0.3333240, 0.3333333) | 0.8172 |
| α | basin 1 start (0.33333, 0.33333) | gap_refine_3d | 2.32e-06 | 1.1272 | (0.3333287, 0.3333310) | 1.0683 |
| β | basin 1 start (0.32338, 0.36190) | gap_refine_3a | 1.49e-04 | 1.0132 | (0.3233800, 0.3619000) | -5.4523 |
| β | basin 1 start (0.32338, 0.36190) | gap_refine_3b | 3.72e-05 | 1.0132 | (0.3233800, 0.3619000) | -0.5912 |
| β | basin 1 start (0.32338, 0.36190) | gap_refine_3c | 9.30e-06 | 0.8767 | (0.3233707, 0.3619279) | 0.4993 |
| β | basin 1 start (0.32338, 0.36190) | gap_refine_3d | 2.32e-06 | 0.8739 | (0.3233730, 0.3619232) | 0.8143 |
| β | basin 1 start (0.32338, 0.36190) | gap_refine_3e | 5.81e-07 | 0.8737 | (0.3233724, 0.3619238) | 0.8687 |
| β | basin 2 start (0.32830, 0.31350) | gap_refine_3a | 1.19e-03 | 6.0869 | (0.3283000, 0.3135000) | -25.9253 |
| β | basin 2 start (0.32830, 0.31350) | gap_refine_3b | 2.98e-04 | 4.3270 | (0.3285975, 0.3137975) | -3.3543 |
| β | basin 2 start (0.32830, 0.31350) | gap_refine_3c | 7.44e-05 | 1.1711 | (0.3284488, 0.3137231) | -0.6960 |
| β | basin 2 start (0.32830, 0.31350) | gap_refine_3d | 1.86e-05 | 0.8822 | (0.3284116, 0.3137231) | 0.4462 |
| β | basin 2 start (0.32830, 0.31350) | gap_refine_3e | 4.65e-06 | 0.8541 | (0.3284209, 0.3137278) | 0.7810 |
| ST | basin 1 start (0.31936, 0.31936) | gap_refine_3a | 5.95e-04 | 0.8648 | (0.3193600, 0.3193600) | -13.6953 |
| ST | basin 1 start (0.31936, 0.31936) | gap_refine_3b | 1.49e-04 | 0.8648 | (0.3193600, 0.3193600) | -2.7471 |
| ST | basin 1 start (0.31936, 0.31936) | gap_refine_3c | 3.72e-05 | 0.8648 | (0.3193600, 0.3193600) | -0.0249 |
| ST | basin 1 start (0.31936, 0.31936) | gap_refine_3d | 9.30e-06 | 0.7348 | (0.3193507, 0.3193507) | 0.5342 |
| ST | basin 1 start (0.31936, 0.31936) | gap_refine_3e | 2.32e-06 | 0.7348 | (0.3193507, 0.3193507) | 0.7145 |

## Wilson-loop WCC for 1H-β (no inversion)

Direct SOC-DFT overlaps (VASP–Wannier90 interface) and Z2Pack on the surface [t, s/2, 0]. Neutral filling is 5 electrons, so the lowest-4 and lowest-6 subspaces are isolated sub-manifolds, not the occupied manifold.

In [6]:
for entry in structures:
    folder = topology / entry["directory"] / "wcc_direct_v3"
    status = load(latest(folder, "status_*.json")) if folder.is_dir() else None
    if status is None:
        continue
    rows = []
    for n, m in status["manifolds"].items():
        gaps = min(stage["minimum_direct_gap_ev"] for stage in m["sampled_gaps"].values())
        rows.append([labels[entry["id"]], n, m["status"], m["z2"], "%.6g" % gaps,
                     len(m["line_positions"]), m["topology_certified"]])
    table(["Structure", "Bands", "Status", "Conditional Z₂", "Min. sampled direct gap (eV)",
           "Wilson loops", "Certified"], rows)
    print(status["status"], "-", status["interpretation"])

| Structure | Bands | Status | Conditional Z₂ | Min. sampled direct gap (eV) | Wilson loops | Certified |
|---|---|---|---|---|---|---|
| 1H-β | 4 | converged_sampled_subspace | 0 | 0.929475 | 23 | False |
| 1H-β | 6 | converged_sampled_subspace | 0 | 0.41504 | 23 | False |

completed_conditional_screening - Conditional even-band subspace analysis assuming electronic time reversal; electronic time-reversal symmetry is not certified.


## Electronic time reversal

Converged SOC state (`tr_evidence.json`): magnetisation density from the noncollinear SCF CHGCAR, TR-odd real part of the one-centre PAW magnetisation (its TR-even imaginary part sets the spin–orbit scale), Kramers splitting at the TRIM, and for 1H-β the E(k)=E(−k) test on the unsymmetrised SCF grid. Stability of the nonmagnetic state (`spin_screen/`): collinear ISPIN=2 SCFs seeded with ferromagnetic, layer-alternating and inversion-odd moments on the same geometry and mesh.

In [7]:
rows, spins = [], []
for entry in structures:
    folder = topology / entry["directory"]
    tr = load(folder / "tr_evidence.json")
    if tr:
        m = tr["magnetisation_density"]
        minus_k = tr.get("scf_minus_k")
        rows.append([labels[entry["id"]], "%.1e" % m["max_abs_m_muB_per_A3"], "%.1e" % m["integral_abs_m_muB"],
                     "%.1e / %.1e" % (m["paw_occupancy_max_abs_re_m"], m["paw_occupancy_max_abs_im_m"]),
                     "%.1e" % tr["trim_kramers_max_splitting_ev"],
                     "%.1e (%d pairs)" % (minus_k["max_abs_dE_ev"], minus_k["pairs"]) if minus_k else "—"])
    spin = load(folder / "spin_screen" / "spin_screen_summary.json")
    if spin is None:
        spins.append([labels[entry["id"]], "pending", "", "", ""])
        continue
    for seed, result in spin["seeds"].items():
        spins.append([labels[entry["id"]], seed, " ".join("%g" % v for v in result["initial_magmom_muB"]),
                      "%.4f" % result["mag_muB"], "%.4f" % max(abs(v) for v in result["site_moments_muB"] or [0]),
                      ])
table(["Structure", "max \\|m(r)\\| (μB/Å³)", "∫\\|m\\| (μB)", "PAW one-centre m: max \\|Re\\| / max \\|Im\\|",
       "TRIM Kramers splitting, bands 1..N+2 (eV)", "max \\|E(k)−E(−k)\\|, bands 1..N+2 (eV)"], rows)
table(["Structure", "Seed", "Initial moments (μB)", "Final total (μB)", "Final max \\|site\\| (μB)"], spins)

| Structure | max \|m(r)\| (μB/Å³) | ∫\|m\| (μB) | PAW one-centre m: max \|Re\| / max \|Im\| | TRIM Kramers splitting, bands 1..N+2 (eV) | max \|E(k)−E(−k)\|, bands 1..N+2 (eV) |
|---|---|---|---|---|---|
| α | 5.9e-08 | 1.6e-07 | 2.3e-07 / 6.7e-05 | 1.5e-07 | — |
| β | 8.9e-08 | 6.9e-07 | 3.7e-08 / 7.3e-05 | 1.7e-07 | — |
| ST | 1.0e-07 | 5.3e-07 | 8.4e-08 / 8.4e-05 | 5.5e-09 | — |
| 2H-α | 5.9e-08 | 2.0e-07 | 6.1e-08 / 7.6e-05 | 1.8e-06 | — |
| 1H-β | 6.1e-08 | 1.6e-07 | 7.9e-08 / 8.4e-05 | 1.2e-06 | 4.0e-07 (5512 pairs) |
| 2H-β | 2.6e-07 | 8.9e-07 | 3.0e-07 / 9.0e-05 | 3.8e-06 | — |

| Structure | Seed | Initial moments (μB) | Final total (μB) | Final max \|site\| (μB) |
|---|---|---|---|---|
| α | pending |  |  |  |
| β | pending |  |  |  |
| ST | pending |  |  |  |
| 2H-α | pending |  |  |  |
| 1H-β | pending |  |  |  |
| 2H-β | pending |  |  |  |